<a href="https://colab.research.google.com/github/pkocmann/messyverse/blob/main/notebooks/05_ki-verschlagwortung.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="In Colab oeffnen"/></a>

# Titel verschlagworten -- mit kontrolliertem Vokabular

Bibliotheken ordnen Titel in **standardisierte Schlagworte** ein. Das Vokabular ist bewusst klein und steht in `katalog/schlagworte.csv`. Dein Auftrag: jedem Titel aus `katalog/titel.csv` genau eine Kategorie aus diesem Vokabular zuordnen.

Diese Aufgabe nutzt **deinen eigenen KI-Assistenten im Chat** -- kein Schluessel, keine Einrichtung im Notebook noetig. Du gibst ihm die Titel und das Vokabular, er schlaegt Kategorien vor, du traegst sie ein und pruefst.

In [ ]:
# SETUP (Black Box -- einmal ausfuehren). Holt deine Arbeitskopie des Uebungsuniversums.
import os, shutil, subprocess
ZIEL = "/content/messyverse"
if os.path.exists(ZIEL):
    shutil.rmtree(ZIEL)              # idempotent: alte Kopie weg, dann frisch klonen
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                "https://github.com/pkocmann/messyverse.git", ZIEL], check=True)
os.chdir(ZIEL)
anzahl = sum(len(d) for r, _, d in os.walk(ZIEL) if ".git" not in r)
print(f"Arbeitskopie: {anzahl} Dateien unter {ZIEL}")

## Schritt 1 -- Titel und Vokabular ansehen

In [ ]:
print("== Vokabular ==")
print(open("katalog/schlagworte.csv").read())
print("== Titel ==")
print(open("katalog/titel.csv").read())

## Schritt 2 -- im Chat verschlagworten lassen

Kopiere die Titel und das Vokabular in deinen KI-Chat und bitte um eine Zuordnung:

> *Hier ist ein kontrolliertes Vokabular (Liste der erlaubten Kategorien) und eine Liste von Buchtiteln. Ordne jedem Titel genau eine Kategorie aus dem Vokabular zu. Gib das Ergebnis als Python-dict aus: ISBN -> Kategorie.*

Trage das Ergebnis des Chats in die naechste Zelle als dict `ergebnis` ein.

In [ ]:
# Zuordnung aus deinem KI-Chat hier als dict eintragen.
# Ziel: dict `ergebnis` -> '<isbn>': '<Kategorie aus dem Vokabular>'
ergebnis = {
    # '9783150012413': 'Philosophie & Ethik',
}


## Schritt 3 -- die Regeln pruefen (Black Box)

Hier gibt es kein einziges richtiges Ergebnis -- Verschlagwortung hat Spielraum. Geprueft wird deshalb gegen **Regeln**: Stammt jede Kategorie aus dem Vokabular? Sind die Pflicht-Facetten besetzt? Sind alle Titel zugeordnet?

In [ ]:
import json, csv
c = json.load(open("loesungen/verschlagwortung.constraints.json"))
vok = set(c["vokabular"])
isbns = {r["isbn"] for r in csv.DictReader(open("katalog/titel.csv"))}
try:
    ergebnis
except NameError:
    ergebnis = {}
fremd = {k: v for k, v in ergebnis.items() if v not in vok}
fehlende_titel = [i for i in isbns if i not in ergebnis]
fehlende_facetten = [f for f in c["pflicht_facetten"] if f not in set(ergebnis.values())]
if not fremd and not fehlende_titel and not fehlende_facetten:
    print(f"OK -- {len(ergebnis)} Titel zugeordnet, alle Regeln erfuellt.")
else:
    if fremd: print("Kategorie nicht im Vokabular:", fremd)
    if fehlende_titel: print("Noch nicht zugeordnet:", fehlende_titel)
    if fehlende_facetten: print("Pflicht-Facette nicht besetzt:", fehlende_facetten)

## Wenn es klemmt -- die Fehlerschleife

Lief der Code auf einen Fehler, oder meldet die Pruefzelle Abweichungen? Dann **nicht selbst reparieren**. Kopiere die Fehlermeldung oder die Abweichungs-Liste, gib sie deinem Assistenten zurueck und bitte um eine korrigierte Fassung. Die haeufigsten Ursachen sind uebersehene Sonderfaelle -- leere Eintraege, fehlende Felder, ungewohnte Schreibweisen.

## Souveraen und ohne Schluessel

Diese Uebung laeuft ueber deinen Chat-Assistenten -- du brauchst keinen API-Schluessel im Notebook. Wer es programmatisch will: ein souveraener EU-Endpoint (z.B. KISSKI/GWDG) eignet sich, fixtures-first aufgebaut. Chinesische Cloud-Endpoints sind ausgeschlossen.